# 21 · Réseaux convolutifs (CNN) — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch21_cnn/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 21.Q1 à 21.Q12, 21.R1 à 21.R3, 21.1 à 21.6, 21.8 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★★ | |
| A | 21.12 à 21.18 | ta librairie conv.py : tailles et padding, la vue « œil de mouche » (im2col), la convolution et le pooling, des tests paramétrés, la chasse aux rayures (*prochaine session de génération*) | | | |
| B | 21.19 à 21.21 | des filtres faits main avec PyTorch, convolutions 1D et 1 × 1, le détecteur de visage hiérarchique (*prochaine session de génération*) | | | |
| C | 21.22 à 21.26 | le convnet MNIST du livre sur Fashion-MNIST, ses courbes, une chasse aux bugs de formes, l'initialisation, stride ou pooling sur CIFAR-10 (*prochaine session de génération*) | | | |
| D | 21.27 à 21.32 | les filtres d'un ResNet-18 pré-entraîné, la montée de gradient sur l'entrée, l'attaque FGSM, la convolution transposée et la rétropropagation dans ta librairie, le défi Fashion-MNIST (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Calculer à la main la taille de sortie, le nombre de paramètres et le champ récepteur d'une pile de convolutions et de poolings.
- Programmer en NumPy, dans `mylearn.conv`, la convolution, `im2col`, le pooling, la convolution transposée et la rétropropagation, et les comparer à `torch.nn.functional`.
- Entraîner des CNN PyTorch, regarder leurs filtres et les tromper par une attaque adversariale.

**Rappel express.** Une couche de convolution calcule une corrélation croisée : $y_{n,o,i,j} = b_o + \sum_{c,u,v} W_{o,c,u,v}\,x_{n,c,\,s i + u,\,s j + v}$, poids de forme $(C_{out}, C_{in}, k_h, k_w)$, $C_{out}(C_{in}k_hk_w + 1)$ paramètres. Taille de sortie : $\lfloor (H + 2p - d(k-1) - 1)/s \rfloor + 1$ ; convolution transposée : $(H - 1)s - 2p + k$. Champ récepteur : $r_\ell = r_{\ell-1} + (k_\ell - 1)\prod_{i<\ell} s_i$. FGSM : $\mathbf{x} + \varepsilon\,\mathrm{sign}(\nabla_{\mathbf{x}}L)$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 21.1 à 21.6 et 🧮 21.8)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une forme ou une position : entre guillemets (`"(3, 5)"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la preuve ∂ 21.7, la réflexion (🗣️ 21.9, ⚖️ 21.10, 📄 21.11) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math

import numpy as np


def out_size(size, k, s=1, p=0, d=1):
    """Output length of a convolution or pooling window (the formula of the sheet)."""
    return (size + 2 * p - d * (k - 1) - 1) // s + 1


def correlate(x, w):
    """Cross-correlation without padding, stride 1 (what nn.Conv2d computes), with plain loops."""
    kh, kw = w.shape
    out = np.zeros((x.shape[0] - kh + 1, x.shape[1] - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(x[i:i + kh, j:j + kw] * w)
    return out


def receptive_field(layers):
    """Side of the receptive field of one output, for a stack of (kernel, stride) layers."""
    r, jump = 1, 1
    for k, s in layers:
        r += (k - 1) * jump
        jump *= s
    return r


# Ex 21.1: the yellow detector R + G - B
YELLOW = np.array([1.0, 1.0, -1.0])

# Ex 21.2: a 5 x 5 image and a Sobel kernel
X_212 = np.array([[3, 1, 0, 0, 1], [2, 0, 1, 2, 0], [4, 1, 0, 1, 3], [1, 2, 2, 0, 1], [0, 3, 1, 2, 2]], float)
K_212 = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], float)
Y_212 = correlate(X_212, K_212)
Y_PAD_212 = correlate(np.pad(X_212, 1), K_212)
CONV_212 = correlate(X_212, K_212[::-1, ::-1])                        # the kernel flipped in both directions
MIN_212 = tuple(int(v) for v in np.unravel_index(np.argmin(Y_212), Y_212.shape))

# Ex 21.4: the MNIST convnet of the book
P_CONV1_214 = 32 * (1 * 9 + 1)
P_CONV2_214 = 64 * (32 * 9 + 1)
P_FC1_214 = 12 * 12 * 64 * 128 + 128
P_FC2_214 = 128 * 10 + 10
P_TOTAL_214 = P_CONV1_214 + P_CONV2_214 + P_FC1_214 + P_FC2_214

# Ex 21.6: transposed convolution, stride 2, as stamps x[i, j] * w with their corner at (2i, 2j)
X_216 = np.array([[2.0, -1.0], [1.0, 3.0]])
W_216 = np.array([[1.0, 0.0, 2.0], [-1.0, 3.0, 0.0], [2.0, 1.0, -1.0]])
Y_216 = np.zeros((5, 5))
for _i in range(2):
    for _j in range(2):
        Y_216[2 * _i:2 * _i + 3, 2 * _j:2 * _j + 3] += X_216[_i, _j] * W_216
Y_PAD_216 = Y_216[1:-1, 1:-1]                                           # padding=1 removes one row/column per border

# Ex 21.8: the compute and the activations of VGG16
VGG_BLOCKS = [[64, 64], [128, 128], [256, 256, 256], [512, 512, 512], [512, 512, 512]]
MACS_208, ACTS_208 = [], 0
_c_in, _side = 3, 224
for _block in VGG_BLOCKS:
    for _c_out in _block:
        MACS_208.append(_side * _side * _c_out * _c_in * 9)
        ACTS_208 += _c_out * _side * _side
        _c_in = _c_out
    _side //= 2
FC_208 = 25088 * 4096 + 4096 * 4096 + 4096 * 1000

**21.Q1 — Les deux sens de « profondeur »**

In [3]:
wb.record("21.Q1a", 3, mistakes={"la profondeur d'une image, ce sont ses canaux, pas sa taille": 224,
          "une photo en couleur a un canal par couleur primaire": 1})
wb.record("21.Q1b", 64, mistakes={"la profondeur d'un tenseur (C, H, W) est son nombre de canaux, le premier nombre": 112})
wb.record("21.Q1c", "B", mistakes={"c'est la profondeur d'un réseau : de quoi est-il fait ?": "A",
          "le nombre de bits par pixel est un autre sens, rare ici": "C",
          "les filtres de VGG16 font 3 × 3": "D"})
wb.record("21.Q1d", 3 * 32 * 32, mistakes={"trois valeurs PAR PIXEL, pas trois features par image : relis l'encadré « Le livre, corrigé » du §21.2": 3,
          "n'oublie pas les trois canaux": 32 * 32,
          "32 × 32 pixels, chacun avec 3 valeurs : un produit, pas une somme": 32 + 32 + 3})
wb.record("21.Q1e", False, mistakes={"le convnet MNIST du livre a aussi un pooling, un dropout et des couches denses : est-ce un CNN ?": True})

WB_ANSWER {"hash": "b8637e54a4dd0e50394eddda0b4d1240a92810cc1cfb076879a45fd6b3d1df33", "id": "21.Q1a", "kind": "int", "magnitude": 0, "mistakes": {"45227156ce53797ec550be076c83382aacc9c33c56cbab373d4417dc901d3fe5": "la profondeur d'une image, ce sont ses canaux, pas sa taille", "467f12570cb7e17fb6f07af3beb09dca67df54644ccb207d9d6ef25b8df20534": "une photo en couleur a un canal par couleur primaire"}, "sign": 1}
📝 Ex 21.Q1a : réponse enregistrée (int).
WB_ANSWER {"hash": "b956dcbd5624ee84e852908e50288a2c67daf5878a2589cc114c1d3db3e3fcf8", "id": "21.Q1b", "kind": "int", "magnitude": 1, "mistakes": {"8dfaf6ce62726160d079c7a8396e4380886aa83ce64b948482472ce1b757e584": "la profondeur d'un tenseur (C, H, W) est son nombre de canaux, le premier nombre"}, "sign": 1}
📝 Ex 21.Q1b : réponse enregistrée (int).
WB_ANSWER {"hash": "ce8df49002acf45a04a219594224b49656ad65c62bd2186878ebf11a21542ea2", "id": "21.Q1c", "kind": "str", "mistakes": {"17e99f0a38c35fcadca3584122cd715f190fce7aecfdbd307e591291baa6

**21.Q2 — Un seul neurone promené sur toute l'image**

In [4]:
wb.record("21.Q2a", True, mistakes={"relis le §21.2.3 : le neurone emporte-t-il ses poids avec lui ?": False})
wb.record("21.Q2b", 25, mistakes={"5 × 5 poids, et un seul canal": 10, "le biais n'est pas demandé": 26})
wb.record("21.Q2c", False, mistakes={"un filtre 5 × 5 a ses 25 poids, sur une image de 100 × 100 comme de 1 000 × 1 000": True})
wb.record("21.Q2d", "B", mistakes={"le noyau, ce sont les poids du filtre": "A",
          "le champ récepteur est tout le carré de pixels lu par le filtre": "C",
          "la carte de caractéristiques est la sortie de la couche": "D"})
wb.record("21.Q2e", "B", mistakes={"un filtre 3 × 3 a plus de poids qu'un 2 × 2 : ce n'est pas la raison": "A",
          "la parité du côté ne change pas la vitesse d'apprentissage": "C",
          "les bibliothèques acceptent des noyaux pairs (padding \"same\" compris)": "D"})

WB_ANSWER {"hash": "3cb2f4d2807cf56879330465285213d128192a46a05204b85b5b094ffcdf4e16", "id": "21.Q2a", "kind": "bool", "mistakes": {"7bc984156e75f7dbd641a80f4f315bcd5165cb3d958b9d5998bad29af6db9f28": "relis le §21.2.3 : le neurone emporte-t-il ses poids avec lui ?"}}
📝 Ex 21.Q2a : réponse enregistrée (bool).
WB_ANSWER {"hash": "e4027ee51d68a9924ee2a3ef188bcbe10fe1b4cfdb921f4e91502657f2278e2d", "id": "21.Q2b", "kind": "int", "magnitude": 1, "mistakes": {"5f2b89d525672fc878a975b5d1c30fb94a2adabb0c235b7ed7e2f42d3d1ae1a5": "5 × 5 poids, et un seul canal", "82df72c6ed227f081a9d88ab8136d62443d16136a5a77eed3db89971b4c08e20": "le biais n'est pas demandé"}, "sign": 1}
📝 Ex 21.Q2b : réponse enregistrée (int).
WB_ANSWER {"hash": "945321951c2295430a9c30f905cd7d0af2f9481a5829100930266f52faaf71c5", "id": "21.Q2c", "kind": "bool", "mistakes": {"13a6a1bc073b03463b3791a20e3959cc05f9451d6a7d67299b5a4783ff502a72": "un filtre 5 × 5 a ses 25 poids, sur une image de 100 × 100 comme de 1 000 × 1 000"}}
📝 Ex 

**21.Q3 — Convolution ou corrélation croisée ?**

In [5]:
wb.record("21.Q3a", False, mistakes={"relis l'encadré 🕰️ du §21.3 : que dit la documentation de PyTorch ?": True})
wb.record("21.Q3b", int(np.dot([1, 2, 3], [1, 0, -1])), mistakes={"c'est la convolution (noyau retourné) : la corrélation croisée ne retourne pas le noyau": 2,
          "la première fenêtre couvre x₀, x₁, x₂ : 1 × 1 + 2 × 0 + 3 × (−1)": -3})
wb.record("21.Q3c", int(np.dot([1, 2, 3], [-1, 0, 1])), mistakes={"c'est la corrélation croisée : la convolution retourne d'abord le noyau": -2})
wb.record("21.Q3d", True, mistakes={"retourner (1, 2, 1) donne… (1, 2, 1)": False})
wb.record("21.Q3e", (6 - 3 + 1) ** 2, mistakes={"(H − k + 1) positions dans chaque direction : un produit des deux": 4,
          "une image 6 × 6 n'a pas 36 morceaux 3 × 3 entiers : le filtre doit tenir dans l'image": 36,
          "ça, c'est pour une image 5 × 5": 9})
wb.record("21.Q3f", 3 * 3 * 3, mistakes={"chaque morceau a k² valeurs PAR CANAL": 9, "3 canaux de 3 × 3 valeurs : un produit": 9 + 3})

WB_ANSWER {"hash": "0df088e482ed54d4fc7c2fd81ad8ce8c04e2e354dd8920c006396bca3b656978", "id": "21.Q3a", "kind": "bool", "mistakes": {"428c0de4f64f987b7557e92592b1066885c3f883b7d93edba3a6fb33b8b11bb0": "relis l'encadré 🕰️ du §21.3 : que dit la documentation de PyTorch ?"}}
📝 Ex 21.Q3a : réponse enregistrée (bool).
WB_ANSWER {"hash": "a1946501ab7c463b97de8fe6e0ff8253f5c6e466401fed336a91ffdfb03bbe33", "id": "21.Q3b", "kind": "int", "magnitude": 0, "mistakes": {"1de0d8e056694c77c1a657a236dc988e478d458831ba385eb6d595e9b37e3f4c": "la première fenêtre couvre x₀, x₁, x₂ : 1 × 1 + 2 × 0 + 3 × (−1)", "e1925df9593bbc74d3eb602add5ee6b67741ecf9abc9433c589a9a17404416fb": "c'est la convolution (noyau retourné) : la corrélation croisée ne retourne pas le noyau"}, "sign": -1}
📝 Ex 21.Q3b : réponse enregistrée (int).
WB_ANSWER {"hash": "e882bb30b2fc39e45b08e2f4b7fc7dda56593d51100cd69c74e36adb78db74dc", "id": "21.Q3c", "kind": "int", "magnitude": 0, "mistakes": {"b3cddce79a4038049a32ea1cef534faa18df16af21

**21.Q4 — Détecteurs de motifs et hiérarchie de filtres**

In [6]:
wb.record("21.Q4a", 3, mistakes={"les pixels noirs (0) ne retirent rien : seuls les blancs sous un −1 font baisser le score": 9,
          "chaque blanc sous un +1 ajoute 1 : combien de +1 y a-t-il ?": 1})
wb.record("21.Q4b", 3 - 1, mistakes={"le pixel du centre est sous un −1 : il retire 1": 3, "la colonne de gauche compte 3 pixels blancs sous des +1": 1})
wb.record("21.Q4c", False, mistakes={"relis la fin du §21.3.1 : qui règle les poids des filtres ?": True})
wb.record("21.Q4d", "B", mistakes={"un filtre 3 × 3 a 9 poids par canal, en haut comme en bas du réseau": "A",
          "le learning rate ne change pas ce que voit un filtre": "C",
          "seule la première couche reçoit les canaux de l'image": "D"})
wb.record("21.Q4e", "B", mistakes={"c'est le sens du ch. 1 ; ici, le livre emploie le mot autrement": "A",
          "un hyperparamètre se règle avant l'entraînement, ce n'est pas ce que cherche un filtre": "C",
          "un filtre cherche une structure faite de plusieurs pixels": "D"})

WB_ANSWER {"hash": "79308843f267d3c32dfb543d38dd4ca9888f490d9da389b61cdd89d877d2aa16", "id": "21.Q4a", "kind": "int", "magnitude": 0, "mistakes": {"a8f6015e4c7e77cdb3c8c35be3146a933340808bb1ebb7eca165b6ff0d13f4e9": "chaque blanc sous un +1 ajoute 1 : combien de +1 y a-t-il ?", "f89b9eac011629d1ff26614a96675ef361315419f683dd25f80251cb3206bc59": "les pixels noirs (0) ne retirent rien : seuls les blancs sous un −1 font baisser le score"}, "sign": 1}
📝 Ex 21.Q4a : réponse enregistrée (int).
WB_ANSWER {"hash": "25b6dba932ebdfb307b9fc0070c9bb8c6b039b653dc414ee7aba53c78d06ab13", "id": "21.Q4b", "kind": "int", "magnitude": 0, "mistakes": {"6c1732f21005fd41d1694c6e0791743599f3537457abbffcbd90500c60abec8b": "le pixel du centre est sous un −1 : il retire 1", "fd8b3cd0283bcdbdf857f97cd16b68842a433607eb7075874bc586292364f373": "la colonne de gauche compte 3 pixels blancs sous des +1"}, "sign": 1}
📝 Ex 21.Q4b : réponse enregistrée (int).
WB_ANSWER {"hash": "5fba711da18caa5cc4098264267634a8e62459420a

**21.Q5 — Padding et stride : que devient la taille ?**

In [7]:
wb.record("21.Q5a", out_size(12, 5), mistakes={"sans padding, on perd k − 1 = 4 pixels, pas k": 7, "sans padding, la sortie est plus petite que l'entrée": 12})
wb.record("21.Q5b", (7 - 1) // 2, mistakes={"(k − 1)/2 de chaque côté, pas k − 1": 6, "un filtre 7 × 7 déborde de 3 pixels de chaque côté": 1})
wb.record("21.Q5c", out_size(7, 3, s=2), mistakes={"⌊(7 − 3)/2⌋ + 1 : n'oublie pas le + 1, et prends le stride vertical (2)": 2})
wb.record("21.Q5d", out_size(12, 3, s=3), mistakes={"la largeur avec le stride horizontal, 3": 5, "⌊(12 − 3)/3⌋ + 1": 3})
wb.record("21.Q5e", True, mistakes={"avec un stride égal au côté du filtre, les fenêtres se recouvrent-elles ?": False})

WB_ANSWER {"hash": "7bdccf6d966e40ee477938ee830e57d7093e14fdbf777747c15901828bcc0e1e", "id": "21.Q5a", "kind": "int", "magnitude": 0, "mistakes": {"164a6e7812e2ca08fc90f8e022e5feb57d433841bb3cac2e86687a6dc7e9c0ee": "sans padding, la sortie est plus petite que l'entrée", "b51d25d83a2aff110b564f2c1e2e413b2c2f27db373d1ba51d6ee7d2b6820016": "sans padding, on perd k − 1 = 4 pixels, pas k"}, "sign": 1}
📝 Ex 21.Q5a : réponse enregistrée (int).
WB_ANSWER {"hash": "22285dc3e8ccc1ab45055a21e379bb133526a3e3a0294c76cc02b62ce8942920", "id": "21.Q5b", "kind": "int", "magnitude": 0, "mistakes": {"4880b6361c4c739481c68870919e0287f7188812c21eab7cb7af11efa7f48f38": "un filtre 7 × 7 déborde de 3 pixels de chaque côté", "4bd91edc9af4b9c3321bf86b8d3683b509c6c4d37bed7a00a357dac37390013a": "(k − 1)/2 de chaque côté, pas k − 1"}, "sign": 1}
📝 Ex 21.Q5b : réponse enregistrée (int).
WB_ANSWER {"hash": "ee74ad1cfd5f87cdeb4ef692f2560319322300441160ecee58ad5c150ff60e47", "id": "21.Q5c", "kind": "int", "magnitude":

**21.Q6 — Canaux d'entrée, nombre de filtres, canaux de sortie**

In [8]:
wb.record("21.Q6a", 6, mistakes={"la profondeur d'un filtre est celle de son ENTRÉE": 4, "un filtre 3 × 3 a une profondeur, en plus": 3})
wb.record("21.Q6b", 4, mistakes={"un canal de sortie par filtre": 6})
wb.record("21.Q6c", 4 * 6 * 3 * 3, mistakes={"chaque filtre a 6 canaux de 3 × 3 poids": 4 * 9, "4 filtres de 6 × 3 × 3 poids": 6 * 9})
wb.record("21.Q6d", 4 * (6 * 3 * 3 + 1), mistakes={"un biais par filtre, pas par poids": 4 * 6 * 9 * 2, "ajoute les 4 biais": 216})
wb.record("21.Q6e", 6 * 3 * 3, mistakes={"par canal : chaque filtre n'a qu'UN canal, et il y en a 6": 6 * 6 * 9,
          "un filtre 3 × 3 par canal, 6 canaux": 9})

WB_ANSWER {"hash": "955ff411eb306d30d371a8471b7753958ee2a4d26a0653752a561d6675718731", "id": "21.Q6a", "kind": "int", "magnitude": 0, "mistakes": {"12fc49c93cf83c0d4e44bb1cadc5df5211d4650bf04df7fd8b1b7a6ede86d91f": "un filtre 3 × 3 a une profondeur, en plus", "d8048fe77f37f35c0d1b6b4ec0598cbe9b0f469835a803a0f8c3434bacea03c7": "la profondeur d'un filtre est celle de son ENTRÉE"}, "sign": 1}
📝 Ex 21.Q6a : réponse enregistrée (int).
WB_ANSWER {"hash": "9c6ec6f9a9cd9572b194ac321a0edd5febb9719a725e9b80c4e8f7fd0c0c4964", "id": "21.Q6b", "kind": "int", "magnitude": 0, "mistakes": {"56ed6d6380cafb3283603803b660332f9c362954197c9073c3ad45c9ad4a1639": "un canal de sortie par filtre"}, "sign": 1}
📝 Ex 21.Q6b : réponse enregistrée (int).
WB_ANSWER {"hash": "e4fd0fa9ee2ff89f81ad61c2c79479a76177008ef96a95f60205fe35b5248b94", "id": "21.Q6c", "kind": "int", "magnitude": 2, "mistakes": {"58ee39dfb8f13107a963be56efe73dcb6f2d88d921de909bf3540f6d1e04d9f7": "chaque filtre a 6 canaux de 3 × 3 poids", "dfb6c2

**21.Q7 — Convolution 1D ou convolution 1×1 ?**

In [9]:
wb.record("21.Q7a", 3 * 50, mistakes={"le filtre couvre toute la largeur : 50 nombres par mot": 3, "3 mots de 50 nombres : un produit": 53})
wb.record("21.Q7b", 20 - 3 + 1, mistakes={"sans padding, le filtre doit tenir entier dans la phrase": 20, "n − k + 1": 17})
wb.record("21.Q7c", 300 * 175, mistakes={"un poids par canal d'entrée et par filtre : un produit": 300 + 175,
          "le biais n'est pas demandé": 300 * 175 + 175})
wb.record("21.Q7d", True, mistakes={"un filtre 1 × 1 lit-il les pixels voisins ?": False})
wb.record("21.Q7e", 28, mistakes={"une convolution 1 × 1 garde la taille de l'image": 175, "la hauteur, pas le nombre de canaux": 300})

WB_ANSWER {"hash": "e749df56b2ad36fba9a2ead3814ba7d88bfe2b8649441b84d575114c197aeb04", "id": "21.Q7a", "kind": "int", "magnitude": 2, "mistakes": {"0ab7a18fda894c366ea5e978affb4fce014ddaebae2ea97138fba93e510292a9": "le filtre couvre toute la largeur : 50 nombres par mot", "dfe81f08ab213342807150afb149be53d11e8a38477a55bf07f6250cf7bcb7d1": "3 mots de 50 nombres : un produit"}, "sign": 1}
📝 Ex 21.Q7a : réponse enregistrée (int).
WB_ANSWER {"hash": "37aabb04a3142975b3a9e08100ac6e8690578cceee9cd7e9ae72cbc65255028f", "id": "21.Q7b", "kind": "int", "magnitude": 1, "mistakes": {"e20b58ed14963faaa91e0e8a01189ed3f99083e83a58c276232b5a731036fe00": "n − k + 1", "eabcc818027f2bf771d48530b6b68875316e9c634145d10241e2593e1b544e28": "sans padding, le filtre doit tenir entier dans la phrase"}, "sign": 1}
📝 Ex 21.Q7b : réponse enregistrée (int).
WB_ANSWER {"hash": "bf58beb7ca2bf6795472f398d593852a571ff1cbef5c3006cfa4123c03806bd1", "id": "21.Q7c", "kind": "int", "magnitude": 4, "mistakes": {"171845c0d9da

**21.Q8 — Couche de convolution et initialisation des filtres**

In [10]:
wb.record("21.Q8a", True, mistakes={"si tout est symétrique au départ, les gradients des filtres sont-ils différents ?": False})
wb.record("21.Q8b", 64 * 3 * 3, mistakes={"le fan-in compte toutes les valeurs reçues : 64 canaux de 3 × 3": 9, "64 canaux × 9 valeurs": 64})
wb.record("21.Q8c", math.sqrt(2 / 576), decimals=3, mistakes={"l'écart-type, pas la variance": 2 / 576,
          "avec un fan-in de 9 seulement, tu as oublié les 64 canaux": math.sqrt(2 / 9),
          "c'est Glorot avec fan-in = fan-out ; He prend √(2/fan-in)": math.sqrt(1 / 576),
          "c'est l'exemple de la fiche, à 32 canaux ; ici, l'entrée en a 64": math.sqrt(2 / 288)})
wb.record("21.Q8d", "B", mistakes={"le noyau, ce sont les poids": "A", "le champ récepteur est ce que lit une sortie": "C",
          "le padding est une bordure ajoutée à l'entrée": "D"})
wb.record("21.Q8e", False, mistakes={"par défaut, padding=0 : que devient une image de 32 avec un noyau 3 ?": True})

WB_ANSWER {"hash": "e7278ed1c6a48ccf9650cfece24b1158d06463e8321610962fbb482b243cd679", "id": "21.Q8a", "kind": "bool", "mistakes": {"d0ea47554d3ef0c63204f662aa573751bfafaace54c95ba017b3da782d42754b": "si tout est symétrique au départ, les gradients des filtres sont-ils différents ?"}}
📝 Ex 21.Q8a : réponse enregistrée (bool).
WB_ANSWER {"hash": "5c37c779e931a1fd26685d1f19e5a1f66df23aef7465863119c4b7da0df861ee", "id": "21.Q8b", "kind": "int", "magnitude": 2, "mistakes": {"872906123432f4155da19f4725fb222db547b5a07e165a55202c1f1956154576": "64 canaux × 9 valeurs", "9407dc72b4d37d1ffc08d1bf9482402b7be65db6494699987915cc933f590399": "le fan-in compte toutes les valeurs reçues : 64 canaux de 3 × 3"}, "sign": 1}
📝 Ex 21.Q8b : réponse enregistrée (int).
WB_ANSWER {"decimals": 3, "hash": "2d08018d6cce300205ee4f898b8b0ef0ba091853a7028c82267d56505163efd8", "hash_coarse": "96a5699a432e904e4ff5b8f903118cad0b938cb1026774d8da69eac506b61b79", "id": "21.Q8c", "kind": "float", "magnitude": -2, "mistakes

**21.Q9 — Convolution transposée : les mots piégés**

In [11]:
wb.record("21.Q9a", False, mistakes={"relis l'encadré « Le livre, corrigé » du §21.8 : que espace la dilatation ?": True})
wb.record("21.Q9b", (4 - 1) * 2 + 3, mistakes={"(H − 1) s − 2p + k, avec p = 0": 8, "la convolution transposée AGRANDIT : (H − 1) s + k": 1})
wb.record("21.Q9c", (4 - 1) * 2 - 2 + 3, mistakes={"le padding retire p cases de CHAQUE côté : − 2p": 8, "sans padding, ce serait 9": 9})
wb.record("21.Q9d", 3 - 1, mistakes={"s − 1 zéros entre deux pixels": 3})
wb.record("21.Q9e", True, mistakes={"relis la fin du §21.8 et sa figure : les tampons se recouvrent-ils de façon égale ?": False})
wb.record("21.Q9f", 3 * (3 - 1) + 1, mistakes={"un noyau k de dilatation d couvre d(k − 1) + 1 pixels": 9, "d(k − 1) + 1 = 3 × 2 + 1": 6})

WB_ANSWER {"hash": "0f6c1a84fb80452adc2ed82db6bd69b5ed36e0c024a86f93017350b0cc89d339", "id": "21.Q9a", "kind": "bool", "mistakes": {"07461db1c8cddb7be403a6ed85d091c689876c6033a28ae8eafe8ce991b9280f": "relis l'encadré « Le livre, corrigé » du §21.8 : que espace la dilatation ?"}}
📝 Ex 21.Q9a : réponse enregistrée (bool).
WB_ANSWER {"hash": "509e165c9a764064cdc7413e21f87185679ca0223f585b9c23fe372769b81690", "id": "21.Q9b", "kind": "int", "magnitude": 0, "mistakes": {"7376d34961ab91fd79c1b7be2438bb8569d6980183324de03d24528044eededb": "(H − 1) s − 2p + k, avec p = 0", "a84ab402a7c2c63f2895059c6978a1d454df5f1428a6bbe815097b335518118d": "la convolution transposée AGRANDIT : (H − 1) s + k"}, "sign": 1}
📝 Ex 21.Q9b : réponse enregistrée (int).
WB_ANSWER {"hash": "5bd857fa0b60c111472543b7dd7133e172a77cb800bff290b0884a69ea281899", "id": "21.Q9c", "kind": "int", "magnitude": 0, "mistakes": {"a695e353d023aa038ffd332d74b09bc0f82e439e254249ee83d6229b6c2ed6c1": "le padding retire p cases de CHAQUE cô

**21.Q10 — VGG16 et ce que voient ses filtres**

In [12]:
wb.record("21.Q10a", "(256, 28, 28)", mistakes={"trois poolings ont divisé 224 par 2³": "(256, 56, 56)",
          "le format (C, H, W) : les canaux d'abord": "(28, 28, 256)",
          "le troisième bloc a 256 filtres": "(128, 28, 28)"})
wb.record("21.Q10b", 5, mistakes={"un pooling par bloc, et VGG16 a cinq blocs": 4})
wb.record("21.Q10c", 224 // 2 ** 4, mistakes={"le cinquième bloc est précédé de quatre poolings, pas cinq": 7,
          "quatre poolings ont divisé 224 par 2⁴": 28})
wb.record("21.Q10d", "B", mistakes={"on ne touche pas aux poids : le réseau n'apprend rien ici": "A",
          "le dropout n'agit pas sur l'image": "C",
          "la méthode part d'une image de bruit, pas des images d'entraînement": "D"})
wb.record("21.Q10e", True, mistakes={"relis le §21.9.2 : que cherchent les filtres de block1, puis ceux de block4 ?": False})

WB_ANSWER {"hash": "24db67087447bc26b8f395a3e29213ee8135dfceaec9e90193c2559da2719322", "id": "21.Q10a", "kind": "str", "mistakes": {"363e81073033aff3b7a833e4ee03ef2f7cb44c8d90ea3ff7bb07f02ad4b9034d": "le format (C, H, W) : les canaux d'abord", "5113b6e76a6eee493bb0fadfcac6e5c4bdf61114ec3b5afb46b8b74879cd31a5": "le troisième bloc a 256 filtres", "c71aa6fc3d6290a129795750fff78ca3d9cf2a7b79be6a3e2078e8205cd2cb5c": "trois poolings ont divisé 224 par 2³"}}
📝 Ex 21.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "065a7b2116ae22b35c5473193eb2340caba04c28ed1ff2fec9bc5fa696c58dfe", "id": "21.Q10b", "kind": "int", "magnitude": 0, "mistakes": {"ce36d3184185ad08732f9a28cca7d355f1541672b3dfb4371f7f727e90c0294a": "un pooling par bloc, et VGG16 a cinq blocs"}, "sign": 1}
📝 Ex 21.Q10b : réponse enregistrée (int).
WB_ANSWER {"hash": "166e85ba89713d9d03ee89c47d7635b920f2b09da2d76a7250ab3700c74d2929", "id": "21.Q10c", "kind": "int", "magnitude": 1, "mistakes": {"0ec3d99fffca401df37bfb4f6a2699531ac8b

**21.Q11 — Exemples adversariaux : vrai ou faux ?**

In [13]:
wb.record("21.Q11a", False, mistakes={"relis le §21.10 : le tigre attaqué ressemble-t-il au tigre d'origine ?": True})
wb.record("21.Q11b", True, mistakes={"relis la définition d'une perturbation universelle au §21.10": False})
wb.record("21.Q11c", False, mistakes={"VGG16, très bien entraîné, est-il trompé par le tigre attaqué ?": True})
wb.record("21.Q11d", 0.03 * 28 * 28, decimals=2, mistakes={"chaque pixel bouge d'exactement ε : il y a 28 × 28 pixels": 0.03,
          "la somme des valeurs absolues des variations, pas leur racine": 0.03 * 28})
wb.record("21.Q11e", "B", mistakes={"une attaque non ciblée cherche n'importe quelle erreur": "A",
          "« universelle » parle d'une perturbation valable pour beaucoup d'images": "C",
          "« physique » parle d'une attaque dans le monde réel (un autocollant)": "D"})

WB_ANSWER {"hash": "66968cd3daa5e60e14edb42d5714aac2b10ac9103e9fe0555f1eed6d089845ed", "id": "21.Q11a", "kind": "bool", "mistakes": {"61966d474ec76def0b5c1091a1dba3a3f62520f58f4318a6656da84259a7fa88": "relis le §21.10 : le tigre attaqué ressemble-t-il au tigre d'origine ?"}}
📝 Ex 21.Q11a : réponse enregistrée (bool).
WB_ANSWER {"hash": "36550db6cbbb0123f3a9ffe2b005db42bb4824b292ef71669bb16c7a17d41a9d", "id": "21.Q11b", "kind": "bool", "mistakes": {"d89fdf2dff312eeca4b065bc95421397880eebb301c23d29d8ae10352137cb0a": "relis la définition d'une perturbation universelle au §21.10"}}
📝 Ex 21.Q11b : réponse enregistrée (bool).
WB_ANSWER {"hash": "22b4e5b0326e69d51ae1980288d98845f0fef6c4b5ba3adaf993a3c5efb41d5f", "id": "21.Q11c", "kind": "bool", "mistakes": {"b06b439937a033782f415d451649fdbf7f0616155f0a371b7c84b6283881abf1": "VGG16, très bien entraîné, est-il trompé par le tigre attaqué ?"}}
📝 Ex 21.Q11c : réponse enregistrée (bool).
WB_ANSWER {"decimals": 2, "hash": "552523c3e6803120ef227dc3f

**21.Q12 — 🕰️ Après VGG16 : ResNet, ViT et modèles pré-entraînés**

In [14]:
wb.record("21.Q12a", "B", mistakes={"ResNet utilise surtout des filtres 3 × 3": "A", "ResNet s'entraîne par rétropropagation": "C",
          "c'est la tête de VGG16 que ResNet a abandonnée": "D"})
wb.record("21.Q12b", (224 // 16) ** 2, mistakes={"224/16 = 14 patchs dans CHAQUE direction": 14, "14 × 14, pas 16 × 16": 256})
wb.record("21.Q12c", 512, mistakes={"le pooling moyen global garde une valeur PAR CANAL": 7 * 7,
          "il ne garde pas les 7 × 7 positions de chaque canal": 512 * 49})
wb.record("21.Q12d", True, mistakes={"relis l'encadré 🕰️ du §21.9.1 : que télécharge weights=\"DEFAULT\" ?": False})

WB_ANSWER {"hash": "6fe4ac75e9127d7d9e4835f093250775c782bb075cb35859fe5e9acaaf9e0389", "id": "21.Q12a", "kind": "str", "mistakes": {"7483f0d1e645fe7a13940bd473d0457930a14e5971b43953c6ac11498547c402": "c'est la tête de VGG16 que ResNet a abandonnée", "a3c93479008b3452ccca6b36795a764624b2a87cff33a055d60100a8d8434afe": "ResNet s'entraîne par rétropropagation", "a69ed8b90e8786120ab6817488aa50463f4908a1cbea6d271be8d120ccadf649": "ResNet utilise surtout des filtres 3 × 3"}}
📝 Ex 21.Q12a : réponse enregistrée (str).
WB_ANSWER {"hash": "8ff39f0de0d17cec3551ac5d1a3276359d7f1785ae4584b62a09775d79801174", "id": "21.Q12b", "kind": "int", "magnitude": 2, "mistakes": {"5f418a023c7f27c1a4bdcb7b46e57d8b3477d390153702325bd5ac8538cca460": "224/16 = 14 patchs dans CHAQUE direction", "98233c487f9e272530a85168fa359d0c3022119200886507e8524da013fc7da8": "14 × 14, pas 16 × 16"}, "sign": 1}
📝 Ex 21.Q12b : réponse enregistrée (int).
WB_ANSWER {"hash": "1ad26cd3f945f53bfab96b567843b9c55d3039084f761ec7930bcec4a9c

**21.R1 — Ce que changent model.train() et model.eval() pour le dropout**

In [15]:
wb.record("21.R1a", True, mistakes={"en évaluation, le dropout inversé ne coupe plus rien et ne multiplie rien": False})
wb.record("21.R1b", 1 / (1 - 0.25), decimals=4, mistakes={"on divise par 1 − p, on ne multiplie pas": 0.75, "on divise par 1 − p, pas par p": 4.0})
wb.record("21.R1c", 25, fractional="en pourcentage : une proportion de 0,25 vaut 25 %", mistakes={"c'est la part GARDÉE": 75})
wb.record("21.R1d", "A", mistakes={"le dropout actif coupe des neurones : peut-il améliorer la mesure ?": "B",
          "le dropout dépend du mode train ou eval": "C", "PyTorch ne vérifie pas le mode pour toi": "D"})

WB_ANSWER {"hash": "b07bcfc3857053d163080b55d2cf5c24e60a1e9e164619ec5c6a4df332438d21", "id": "21.R1a", "kind": "bool", "mistakes": {"0743f07040248237749a8af5673f1b9a47c4fd9ad70061c74f30fbdf1e3695b9": "en évaluation, le dropout inversé ne coupe plus rien et ne multiplie rien"}}
📝 Ex 21.R1a : réponse enregistrée (bool).
WB_ANSWER {"decimals": 4, "hash": "baefb7a288a88fdd30fc7850b8fa22a0e85305bcd71a8c6ed80881aefdf721f6", "hash_coarse": "f5ecdfd42acf53d0a2c26d4a91714be268d0628060d1612fdd429b900ebfcf93", "id": "21.R1b", "kind": "float", "magnitude": 0, "mistakes": {"2ba9ff806787f4c26c1b3eba151d2df6840e7cf54dc175c6d18352334b61ec64": "on divise par 1 − p, on ne multiplie pas", "57777c3ff7cec3db600cfb9c0b7464e81ba1a56a323d8479e2e5505b48bcb65c": "on divise par 1 − p, pas par p"}, "sign": 1}
📝 Ex 21.R1b : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"fractional": "en pourcentage : une proportion de 0,25 vaut 25 %", "hash": "cb2ff2bc98310a99fb698d6744962f63ba64aebb604e11fb2b4770b04df48b

**21.R2 — Le gradient d'un poids utilisé à plusieurs endroits**

In [16]:
wb.record("21.R2a", 1 + 2 + 3, mistakes={"chaque y_i dépend de w avec le coefficient x_i : additionne les x_i, pas les ∂L/∂y_i seuls (qui valent 1)": 3})
wb.record("21.R2b", 1 * 1 + (-1) * 2 + 2 * 3, mistakes={"chaque contribution est (∂L/∂y_i) × x_i : n'oublie pas le signe −": 9,
          "somme des (∂L/∂y_i) × x_i, pas des ∂L/∂y_i seuls": 2})
wb.record("21.R2c", True, mistakes={"relis la fiche du ch. 18 : `+=` ou `=` dans backward ?": False})

WB_ANSWER {"hash": "d962252d1a9c14b5f087e73d39ca643c3becf85c711530aa65d941675027b2ee", "id": "21.R2a", "kind": "int", "magnitude": 0, "mistakes": {"76d39d81c8c33fc8bcaacc2e835cd66f17cb5b7573cdd4a071cec5c219e983b1": "chaque y_i dépend de w avec le coefficient x_i : additionne les x_i, pas les ∂L/∂y_i seuls (qui valent 1)"}, "sign": 1}
📝 Ex 21.R2a : réponse enregistrée (int).
WB_ANSWER {"hash": "d9d59ae9a1898d4d53d9203dff9da456e61dc303addd24e22e2639baccd05208", "id": "21.R2b", "kind": "int", "magnitude": 0, "mistakes": {"8c868c92dcf2b61326a8853710c2fe4d3eb5c509c3e16869a8207a8571dff060": "somme des (∂L/∂y_i) × x_i, pas des ∂L/∂y_i seuls", "abc46c1cedd6134e134901504acedc704ea3b498a98d4af1b1160550e03e3650": "chaque contribution est (∂L/∂y_i) × x_i : n'oublie pas le signe −"}, "sign": 1}
📝 Ex 21.R2b : réponse enregistrée (int).
WB_ANSWER {"hash": "7a907bfc7fbedc04a9abd9155611541e00a5e67ed21223a9e066f90eaa14c050", "id": "21.R2c", "kind": "bool", "mistakes": {"b965c23da68380201500cfaa671dc216a

**21.R3 — Le produit scalaire, ce que calcule un filtre en chaque position**

In [17]:
wb.record("21.R3a", int(np.dot([1, 2, -1], [3, 0, 2])), mistakes={"n'oublie pas le signe de −1 × 2": 5, "la somme des produits terme à terme": 6})
wb.record("21.R3b", 1 * 0 + 2 * 1 + 3 * 1 + 4 * 0, mistakes={"terme à terme, pas un produit matriciel": 10, "les cases hors diagonale (2 et 3) comptent : la seconde grille y vaut 1": 0})
wb.record("21.R3c", True, mistakes={"w · x = ‖w‖ ‖x‖ cos θ : quand est-il le plus grand ?": False})

WB_ANSWER {"hash": "04eff85be8a91779bc2574afcdf256531c5b47da806ae18079d14176f6c6abf6", "id": "21.R3a", "kind": "int", "magnitude": 0, "mistakes": {"2c4893a12390e817bf651431aa707e5f1b9249383af05c1c5e062bdbb0f9f876": "la somme des produits terme à terme", "494040d4938774fd4c7dee6adbd40c2c8264527a18edf92fd14c23f79508c380": "n'oublie pas le signe de −1 × 2"}, "sign": 1}
📝 Ex 21.R3a : réponse enregistrée (int).
WB_ANSWER {"hash": "d67963992a1ce141cd9e25dadce659494a2a13e0a736d9901958eefba56ca003", "id": "21.R3b", "kind": "int", "magnitude": 0, "mistakes": {"a0a3ed4d69c6be3cb6507e8a29d7691ba65fd50f0667a848e47ef3db40b91e2f": "terme à terme, pas un produit matriciel", "d93407c616085630d5ac5a8d7f4e1edd9a699a295594a03685f14c090ae59518": "les cases hors diagonale (2 et 3) comptent : la seconde grille y vaut 1"}, "sign": 1}
📝 Ex 21.R3b : réponse enregistrée (int).
WB_ANSWER {"hash": "1f0deba1fca8f60c5771055734e990233251f72c87d892fb01c3feb7db0f5b21", "id": "21.R3c", "kind": "bool", "mistakes": {"2d4

**Ex 21.1 — Le détecteur de jaune à la main**

In [18]:
wb.record("21.1a", float(YELLOW @ [0.9, 0.85, 0.1]), decimals=2, mistakes={"le bleu compte avec le poids −1 : R + G − B, pas R + G + B": 1.85})
wb.record("21.1b", float(YELLOW @ [1.0, 0.5, 0.0]), decimals=2, mistakes={"R + G − B avec B = 0": 0.5})
wb.record("21.1c", float(YELLOW @ [1.0, 1.0, 1.0]), decimals=2, mistakes={"le bleu du blanc retire 1": 2, "1 + 1 − 1": 3})
wb.record("21.1d", 0.0, decimals=2, mistakes={"un pixel noir a ses trois composantes à 0": 1})
wb.record("21.1e", "A", mistakes={"compare les scores de a) à d) : lequel est le plus grand ?": "B",
          "le blanc a du bleu, que le neurone retire": "C", "le noir a un score nul": "D"})
wb.record("21.1f", 2.0, decimals=2, mistakes={"le maximum est atteint pour R = G = 1 et B = 0": 3, "avec B = 0, pas B = 1": 1})
wb.record("21.1g", -1, mistakes={"le magenta n'a pas de vert : le vert doit faire baisser le score": 1, "le vert doit compter, en négatif": 0})
wb.record("21.1h", 4 * 6, mistakes={"un score par pixel, pas par valeur (le neurone combine les trois canaux)": 4 * 6 * 3,
          "4 × 6 pixels": 10})
wb.record("21.1i", True, mistakes={"un neurone qui lit les 3 canaux d'un seul pixel, avec les mêmes poids partout : relis le §21.6": False})

WB_ANSWER {"alt_coarse": ["b307cbeaefe4abaeba5bcca8d30fb195b211a519c67f324cfb3d9bcd161537c7"], "decimals": 2, "hash": "e9839765d46bd5b1ce8401bda92b839f417df5f7d691121926875c8d7f73ace7", "hash_coarse": "b2fea84afd949742e845daffba29acf8e705f6046fabd48e402c61c1b2c7eea2", "id": "21.1a", "kind": "float", "magnitude": 0, "mistakes": {"e76ec0dddfe74615d9f1637039c0c3b8788a4fa70e8388228e70b52768771783": "le bleu compte avec le poids −1 : R + G − B, pas R + G + B"}, "sign": 1}
📝 Ex 21.1a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "b6dd5980e0c2ea913e31d547bb78dc3cdf3ec050e1eef3e8f406dbb68a1308ca", "hash_coarse": "6b00045604a600991def8b272b20ca4e71866a94e08bcd11bebf8392598da668", "id": "21.1b", "kind": "float", "magnitude": 0, "mistakes": {"6726af78efabd35701d0d126e9ea88ba583f8b0f84a7081474526e393a29106d": "R + G − B avec B = 0"}, "sign": 1}
📝 Ex 21.1b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "e8570d4051c4624792dc14c255f44

**Ex 21.2 — Une corrélation 5×5 par 3×3 à la main**

In [19]:
wb.record("21.2a", Y_212.shape[1], mistakes={"sans padding, un filtre 3 × 3 fait perdre 2 pixels, pas 3": 2, "sans padding, la sortie rétrécit": 5})
wb.record("21.2b", int(Y_212[0, 0]), mistakes={"c'est la valeur avec le noyau retourné (convolution) : nn.Conv2d ne retourne pas le noyau": int(CONV_212[0, 0]),
          "le noyau a des 2 sur sa ligne du milieu": int(np.sum(X_212[:3, :3] * np.array([[1, 0, -1]] * 3)))})
wb.record("21.2c", int(Y_212[1, 1]), mistakes={"le morceau dont le coin en haut à gauche est en (1, 1) : lignes 1 à 3, colonnes 1 à 3": int(Y_212[0, 0])})
wb.record("21.2d", int(Y_212[2, 1]), mistakes={"ligne 2, colonne 1 : vérifie que tu n'as pas pris (1, 2)": int(Y_212[1, 2]),
          "c'est la valeur avec le noyau retourné": int(CONV_212[2, 1])})
wb.record("21.2e", int(Y_212.min()), mistakes={"la plus PETITE valeur, pas la plus grande": int(Y_212.max())})
wb.record("21.2f", "(1, 2)", mistakes={"ligne puis colonne : tu les as inversées": "(2, 1)",
          "c'est la position de la plus grande valeur": "(0, 0)"})
wb.record("21.2g", int(Y_PAD_212[0, 0]), mistakes={"avec le padding, la position (0, 0) est centrée sur le pixel X[0, 0], entouré de zéros": int(Y_212[0, 0])})
wb.record("21.2h", int(Y_PAD_212[4, 0]), mistakes={"la position (4, 0) : la dernière ligne, la première colonne": int(Y_PAD_212[0, 4])})
wb.record("21.2i", int(CONV_212[0, 0]), mistakes={"la convolution retourne le noyau : ce n'est pas la valeur de b)": int(Y_212[0, 0])})
wb.record("21.2j", True, mistakes={"retourne K dans les deux directions : que vaut le noyau retourné, comparé à K ?": False})

WB_ANSWER {"hash": "fbd5d8b28387662cf5e5fcaca57c44ba73e55ece1973d63f1d0d7c5b04e5f52f", "id": "21.2a", "kind": "int", "magnitude": 0, "mistakes": {"d0409ba5412b1725899c67c17c236627a30a7b50423970ddb9417453986a5722": "sans padding, la sortie rétrécit", "e858c96a00a807da12b1cfd13489ed83ed7b54557260320c6ad4970c41379565": "sans padding, un filtre 3 × 3 fait perdre 2 pixels, pas 3"}, "sign": 1}
📝 Ex 21.2a : réponse enregistrée (int).
WB_ANSWER {"hash": "ec99a7d2b4d04ad9f6a749fd15af1a48bd7ccb03cc316703b9714a1666ac8ace", "id": "21.2b", "kind": "int", "magnitude": 0, "mistakes": {"b7d82b503c0cf2f67892b760d67d177adecf8be7d5eacece9d84fad6e7258f25": "c'est la valeur avec le noyau retourné (convolution) : nn.Conv2d ne retourne pas le noyau", "f3fd87b72eb9e3d2ba7e5d602022b2238a4683c94328a0babe90d9d332ab1eb4": "le noyau a des 2 sur sa ligne du milieu"}, "sign": 1}
📝 Ex 21.2b : réponse enregistrée (int).
WB_ANSWER {"hash": "becf89185ca6f007bed6b1f79513ff36ecc43e1d39e7e6aac66cffa0975fd0a4", "id": "21.2c

**Ex 21.3 — Tailles de sortie : padding, stride et la hiérarchie 600 → 300 → 100**

In [20]:
wb.record("21.3a", out_size(600, 5, s=2, p=2), mistakes={"⌊(600 + 4 − 5)/2⌋ + 1 : n'oublie pas le + 1": 299})
wb.record("21.3b", out_size(300, 3, s=3, p=2), mistakes={"c'est ce qu'annonce le livre : applique la formule": 100,
          "⌊·⌋ arrondit vers le bas : ⌊301/3⌋ = 100, puis + 1": 102})
wb.record("21.3c", 0, mistakes={"essaie p = 0 : ⌊(300 − 3)/3⌋ + 1": 1, "avec p = 2, on trouve b)": 2})
wb.record("21.3d", 1, mistakes={"avec p = 2, on trouve b), pas 100": 2, "p = 1 donne-t-il encore 100 ?": 0})
wb.record("21.3e", 8, mistakes={"c'est ce qu'écrit le livre ; la profondeur d'un filtre est celle de son entrée": 4,
          "l'entrée du second étage est la sortie du premier, qui a 8 filtres": 1})
wb.record("21.3f", out_size(28, 3, d=2), mistakes={"le noyau dilaté couvre d(k − 1) + 1 = 5 pixels, pas 3": 26, "il couvre 5 pixels, pas 6": 23})
wb.record("21.3g", 2, mistakes={"le noyau dilaté couvre 5 pixels : il déborde de 2 de chaque côté": 1, "la bordure totale, 4, se partage en deux": 4})
wb.record("21.3h", (100 - 1) * 3 + 3, mistakes={"(H − 1) s + k, pas H × s + k": 303, "c'est la formule d'une convolution ordinaire : une convolution transposée agrandit": 33})
wb.record("21.3i", (7 - 1) * 2 - 2 * 1 + 4, mistakes={"− 2p : un de chaque côté": 15, "(H − 1) s − 2p + k = 12 − 2 + 4": 16})
wb.record("21.3j", out_size(14, 4, s=2, p=1), mistakes={"vérifie le padding (+ 2p) et le + 1 : ⌊(14 + 2 − 4)/2⌋ + 1": 6, "il manque le padding (+ 2) et le + 1": 5})

WB_ANSWER {"hash": "a3d7380ed3567e54759c5cea32705daaa9098b1dec321f31678b6f3bcf2352fe", "id": "21.3a", "kind": "int", "magnitude": 2, "mistakes": {"2b4ec7af3ab8e07363909501acecbfd37c38f86aaf71d94d5283409de1f3b216": "⌊(600 + 4 − 5)/2⌋ + 1 : n'oublie pas le + 1"}, "sign": 1}
📝 Ex 21.3a : réponse enregistrée (int).
WB_ANSWER {"hash": "3e016e60a7d54835db3af692383db9ebae058fd3ecf2c14273da9199d406a7f1", "id": "21.3b", "kind": "int", "magnitude": 2, "mistakes": {"b73824d16799432b54285fdf9a405407ac91694caab74fc9b38b211123625793": "⌊·⌋ arrondit vers le bas : ⌊301/3⌋ = 100, puis + 1", "e4cc9000b2665a2ae9d1135acca5f0955653b5ef6289a19bd12de484c9110bf4": "c'est ce qu'annonce le livre : applique la formule"}, "sign": 1}
📝 Ex 21.3b : réponse enregistrée (int).
WB_ANSWER {"hash": "ed97b25a927663130f827b65c26aabe7da773e426605aa5bf8d56980c5b5cd1d", "id": "21.3c", "kind": "int", "magnitude": null, "mistakes": {"aed2ce161784bff1dcb994b71a968d94fdf39e19a1ccf38e87a4bdc42f37c63e": "avec p = 2, on trouve b)", 

**Ex 21.4 — Compter les paramètres : convolution contre couche dense**

In [21]:
wb.record("21.4a", P_CONV1_214, mistakes={"ajoute un biais par filtre": 32 * 9, "32 filtres de 3 × 3 poids sur un canal, plus un biais chacun": 9 + 1})
wb.record("21.4b", P_CONV2_214, mistakes={"chaque filtre a 32 canaux de 3 × 3 poids": 64 * 10, "ajoute les 64 biais": 64 * 32 * 9})
wb.record("21.4c", P_FC1_214, mistakes={"ajoute les 128 biais": 9216 * 128, "l'entrée est l'aplatissement de 64 × 12 × 12 = 9 216 valeurs": 64 * 128 + 128})
wb.record("21.4d", P_FC2_214, mistakes={"ajoute les 10 biais": 1280})
wb.record("21.4e", P_TOTAL_214, mistakes={"n'oublie pas la dernière couche dense : additionne a) à d)": P_TOTAL_214 - P_FC2_214,
          "ajoute les biais de chaque couche": P_TOTAL_214 - 32 - 64 - 128 - 10})
wb.record("21.4f", 100 * P_FC1_214 / P_TOTAL_214, decimals=1, mistakes={"la part de la première couche dense, pas celle des convolutions": 100 * (P_CONV1_214 + P_CONV2_214) / P_TOTAL_214})
wb.record("21.4g", 784 * 32 * 26 * 26 + 32 * 26 * 26, mistakes={"ajoute un biais par sortie : 32 × 26 × 26": 784 * 32 * 26 * 26,
          "une couche dense relie chaque entrée (784) à chaque sortie (32 × 26 × 26)": 32 * 26 * 26})
wb.record("21.4h", 26 * 26 * 32 * 9, mistakes={"9 multiplications par position et par filtre, sur les 26 × 26 positions": 26 * 26 * 9,
          "le nombre de positions de SORTIE, 26 × 26, pas 28 × 28": 28 * 28 * 32 * 9})

WB_ANSWER {"hash": "684bdc61479757a6b236f85ed065bafda6a8cc3036859c51e979449f4985bf3b", "id": "21.4a", "kind": "int", "magnitude": 2, "mistakes": {"6903b6c5d3dad3104d40c30becde887094ab27a712cf3bd80e3db74163712fbe": "32 filtres de 3 × 3 poids sur un canal, plus un biais chacun", "b3a1dbfe7e5431f41e975d10e37309e50364c414b621c3eb792b131d08ed6544": "ajoute un biais par filtre"}, "sign": 1}
📝 Ex 21.4a : réponse enregistrée (int).
WB_ANSWER {"hash": "75e539e55988e655a74d5f7272e3bc5976492a9e534e6786f17a1b69e90081a3", "id": "21.4b", "kind": "int", "magnitude": 4, "mistakes": {"1ce0a86de2ecf78ac897e087849ae323986de4df0afcd31f4afcf0f3e3c40da0": "chaque filtre a 32 canaux de 3 × 3 poids", "586bf95429eb9fd8ad0683ba042b4da165dfd0bd742345ab5abf4d1a1237467e": "ajoute les 64 biais"}, "sign": 1}
📝 Ex 21.4b : réponse enregistrée (int).
WB_ANSWER {"hash": "93965623bfa38468395db64c72878cf425082949831969b17e57fffdc82c3f1f", "id": "21.4c", "kind": "int", "magnitude": 6, "mistakes": {"609bfe6c909a72d69907a480

**Ex 21.5 — Champ récepteur effectif d'une pile convolution + pooling**

In [22]:
wb.record("21.5a", receptive_field([(3, 1), (3, 1)]), mistakes={"chaque couche ajoute k − 1 = 2 pixels, pas 3": 6, "3 + 2": 3})
wb.record("21.5b", receptive_field([(3, 1)] * 3), mistakes={"1 + 2 + 2 + 2": 9})
wb.record("21.5c", 3 * 9 * 64 * 64, mistakes={"trois couches de 64 × 64 × 3 × 3 poids": 9 * 64 * 64, "le biais n'est pas demandé": 3 * (9 * 64 + 1) * 64})
wb.record("21.5d", 7 * 7 * 64 * 64, mistakes={"le champ récepteur de b) est 7 × 7 : une seule convolution 7 × 7": 5 * 5 * 64 * 64})
wb.record("21.5e", receptive_field([(3, 1), (3, 3), (3, 1)]), mistakes={"c'est le compte du livre : il oublie le débord des filtres du niveau 1": 9,
          "après le pooling, deux cases voisines sont espacées de 3 pixels : le dernier noyau ajoute (3 − 1) × 3": 7})
wb.record("21.5f", receptive_field([(3, 1), (3, 3), (3, 1), (3, 3), (3, 1)]), mistakes={"après le second pooling, deux cases voisines sont espacées de 9 pixels": 23,
          "le dernier noyau ajoute (3 − 1) × 9, et le pooling d'avant (3 − 1) × 3": 27})
wb.record("21.5g", receptive_field([(5, 2), (3, 3)]), mistakes={"c'est l'approximation du livre (6 × 6) ; le second noyau ajoute (3 − 1) × 2 au premier": 6,
          "le premier stride, 2, multiplie le débord du second noyau": 7})
wb.record("21.5h", receptive_field([(3, 1), (3, 1), (2, 2)]), mistakes={"le pooling 2 × 2 compte comme une couche de noyau 2": 5, "2 + 2 + 1, partant de 1": 7})

WB_ANSWER {"hash": "3938c5b574070257d3d778dcbb298be6e31f2c529db9f4d4de4f79a5fe825d34", "id": "21.5a", "kind": "int", "magnitude": 0, "mistakes": {"4837323b6319897d2ea014782500832f3f8b9ea7361459f1df15b642447357b2": "chaque couche ajoute k − 1 = 2 pixels, pas 3", "9922fc945fde7575f50a960cb7f00c06d8f174699929610a2d599d6ab634a03c": "3 + 2"}, "sign": 1}
📝 Ex 21.5a : réponse enregistrée (int).
WB_ANSWER {"hash": "2d95a2982a1565575c1fb5192b7fd506d14cdad6aebefea74c8c967d23621a87", "id": "21.5b", "kind": "int", "magnitude": 0, "mistakes": {"dcda77efe5eb0bb3855e751a77962f062ff32dc723fd06513538bb9e2ba5ae8c": "1 + 2 + 2 + 2"}, "sign": 1}
📝 Ex 21.5b : réponse enregistrée (int).
WB_ANSWER {"hash": "5df01ec6075c994c9fe9051381cc4dea61cdc295aa5157f05542ec8f45b5cc04", "id": "21.5c", "kind": "int", "magnitude": 5, "mistakes": {"90d19384cbb222015c2f58c1e5f1981321574d5d1c44115757053d05dc25a67d": "le biais n'est pas demandé", "a4a685412ab781c586ae371d6f0da9c5fee0e03f2775c2859ad7a4e33b8ec559": "trois couches

**Ex 21.6 — Convolution transposée à la main : insérer des zéros puis corréler**

In [23]:
wb.record("21.6a", Y_216.shape[1], mistakes={"(H − 1) s + k = 1 × 2 + 3": 4, "les tampons se chevauchent : (H − 1) s + k = 1 × 2 + 3, pas 2 × 3": 6})
wb.record("21.6b", 2 - 1, mistakes={"s − 1 zéros entre deux pixels": 2})
wb.record("21.6c", 3 - 1 - 0, mistakes={"k − 1 − p, avec k = 3 et p = 0": 1, "k − 1 − p = 2": 3})
wb.record("21.6d", int(Y_216[0, 0]), mistakes={"seul le tampon de x[0, 0] = 2 couvre la case (0, 0)": 1})
wb.record("21.6e", int(Y_216[1, 1]), mistakes={"seul le tampon de x[0, 0] couvre la case (1, 1) : x[0, 0] × w[1, 1]": int(X_216.sum() * W_216[1, 1])})
wb.record("21.6f", int(Y_216[0, 2]), mistakes={"la case (0, 2) reçoit DEUX tampons : celui de x[0, 0] et celui de x[0, 1]": 4,
          "le tampon de x[0, 1] commence en (0, 2) : il y dépose x[0, 1] × w[0, 0]": -1})
wb.record("21.6g", int(Y_216[2, 2]), mistakes={"la case (2, 2) reçoit les QUATRE tampons, celui de x[1, 1] compris": int(X_216[0, 0] * W_216[2, 2] + X_216[0, 1] * W_216[2, 0] + X_216[1, 0] * W_216[0, 2])})
wb.record("21.6h", int(Y_216.sum()), mistakes={"chaque tampon a pour somme x[i, j] × (somme de w)": int(X_216.sum()),
          "la somme de y est (somme de x) × (somme de w)": int(W_216.sum())})
wb.record("21.6i", Y_PAD_216.shape[1], mistakes={"− 2p : on retire une colonne de CHAQUE côté": 4})
wb.record("21.6j", int(Y_PAD_216[1, 1]), mistakes={"le centre de la sortie 3 × 3 est la case (2, 2) de la sortie sans padding": int(Y_216[1, 1])})

WB_ANSWER {"hash": "f3dbae55194954e9950f0371e0883a79dc3c7b55a9e7798e4903197a83d59902", "id": "21.6a", "kind": "int", "magnitude": 0, "mistakes": {"7163e65c74c0e7412cf9e387043282eb972898a3cf3fd406fcf6826d74eac3a3": "(H − 1) s + k = 1 × 2 + 3", "8a4cc85ddf82869d98b8d044f8674654e1ba5f761aea833ae3612d1cb5fa6c47": "les tampons se chevauchent : (H − 1) s + k = 1 × 2 + 3, pas 2 × 3"}, "sign": 1}
📝 Ex 21.6a : réponse enregistrée (int).
WB_ANSWER {"hash": "c7d49223c307412be0b5ddc414d0ad27b06ae5745ee24441c1c76331391fa8d1", "id": "21.6b", "kind": "int", "magnitude": 0, "mistakes": {"8b44e595124d662344f8c2241f0c0d276a9e1e32e2954d8c8b2d267dce205c9a": "s − 1 zéros entre deux pixels"}, "sign": 1}
📝 Ex 21.6b : réponse enregistrée (int).
WB_ANSWER {"hash": "05e395545cc0638b83ce5b9248174b75e8284d8567564222f7305d59448be401", "id": "21.6c", "kind": "int", "magnitude": 0, "mistakes": {"d331fe7a3f1563094492ef2043a5dd75c8d6a2ccef30cc57ac78f04f4ac3c897": "k − 1 − p = 2", "f6ea521654d3f83433512f915e59a1cce7ea2

**Ex 21.8 — Fermi : paramètres, mémoire et calcul de VGG16**

In [24]:
wb.record("21.8a", MACS_208[0] / 1e6, decimals=1, mistakes={"224 × 224 positions, 64 filtres, 3 × 3 × 3 = 27 poids chacun": 224 * 224 * 64 * 9 / 1e6,
          "en millions": MACS_208[0] / 1e9})
wb.record("21.8b", MACS_208[1] / 1e9, decimals=2, mistakes={"chaque filtre a 64 canaux de 3 × 3 poids": 224 * 224 * 64 * 9 / 1e9,
          "en milliards": MACS_208[1] / 1e6})
wb.record("21.8c", sum(MACS_208) / 1e9, decimals=1, mistakes={"chaque pooling divise par 2 le côté de l'image des blocs suivants": sum(c_in * c_out * 224 * 224 * 9 for c_in, c_out in [(3, 64), (64, 64), (64, 128), (128, 128), (128, 256), (256, 256), (256, 256), (256, 512), (512, 512), (512, 512), (512, 512), (512, 512), (512, 512)]) / 1e9,
          "chaque filtre a C_in canaux de 3 × 3 poids (et le côté est divisé par 2 à chaque pooling)": sum(c * 224 * 224 * 9 for c in [64, 64, 128, 128, 256, 256, 256, 512, 512, 512, 512, 512, 512]) / 1e9})
wb.record("21.8d", 100 * FC_208 / (FC_208 + sum(MACS_208)), decimals=1, mistakes={"la part des couches denses dans le CALCUL, pas dans les paramètres": 89.4})
wb.record("21.8e", 4 * ACTS_208 / 1e6, decimals=1, mistakes={"4 octets par valeur float32": ACTS_208 / 1e6, "en Mo de 10⁶ octets": 4 * ACTS_208 / 2 ** 20})
wb.record("21.8f", 64 * 4 * ACTS_208 / 1e9, decimals=2, mistakes={"multiplie par les 64 images du mini-batch": 4 * ACTS_208 / 1e9,
          "Go de 10⁹ octets, pas Gio": 64 * 4 * ACTS_208 / 2 ** 30})
wb.record("21.8g", True, mistakes={"compare ta réponse de d) à la part des paramètres des couches denses (✏️ 20.6)": False})

WB_ANSWER {"decimals": 1, "hash": "792a0e0a5b9b313968af36bcc0ed0858e5652629fab5872947f49ba21c7421de", "hash_coarse": "eea7d6f20bebd60696f229575cfc70ccf0139051fb88a6688f49c625856293b3", "id": "21.8a", "kind": "float", "magnitude": 1, "mistakes": {"19985e9ab76457f22272f72907b36671f4a02c81b1581ec5bfc660bb176c744e": "224 × 224 positions, 64 filtres, 3 × 3 × 3 = 27 poids chacun", "5cded53f9d832ddd2eed2bcce4a4703941fbddd4d93f2d319391c38b8b8a4aa1": "en millions"}, "sign": 1}
📝 Ex 21.8a : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "994e6a38fee7f8cb3389db92ebe1f1d17fabb9944383da53e9d010f5da410311", "hash_coarse": "ce403f2e12f5d87fe40f554ad554bc017a734cb09c8fa62953f21f09baf4beb8", "id": "21.8b", "kind": "float", "magnitude": 0, "mistakes": {"00de5416c34eb3b4e1a1b59019ec47976b4b920052d187dc05f26cb089e48c49": "en milliards", "f56c4e7b2ce65f8c798d07240d455245a93185d1fdb51b6bc1158cfe510c08d6": "chaque filtre a 64 canaux de 3 × 3 poids"}, "sign": 1}
📝 Ex 21.8b : rép

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer la forme de sortie et le nombre de paramètres d'une couche de convolution, avec padding, stride et dilatation ?
2. Sais-tu expliquer le partage de poids, le champ récepteur et ce que font une convolution 1 × 1 et une convolution transposée ?
3. Sais-tu dire comment on regarde ce que voit un filtre, et comment on fabrique un exemple adversarial ?

**Pour aller plus loin** : les notes de CS231n sur les réseaux convolutifs et le guide de Dumoulin et Visin, cités dans la fiche. La suite : le ch. 22, les réseaux récurrents.

*Les exercices de code 21.12 à 21.32 (parties A à D, ta librairie `conv.py` et les CNN PyTorch) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 21` pour obtenir le notebook complet.*